In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Latihan4") \
    .master("local[2]") \
    .config("spark.driver.memory", "512m") \
    .config("spark.driver.maxResultSize", "256m") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
print("Siap. Jumlah baris:", df.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 19:17:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Siap. Jumlah baris: 600


In [2]:
df.filter(df["metode_pembayaran"] == "E-Wallet") \
  .select("order_id", "kategori", "total_pendapatan") \
  .show()

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1001|        Rumah Tangga|         1250000|
|ORD-1003|        Rumah Tangga|         2500000|
|ORD-1013|   Makanan & Minuman|          200000|
|ORD-1014|             Fashion|           50000|
|ORD-1016|        Rumah Tangga|         2000000|
|ORD-1017|   Makanan & Minuman|         3000000|
|ORD-1020|   Makanan & Minuman|          100000|
|ORD-1023|        Rumah Tangga|           75000|
|ORD-1037|          Elektronik|          750000|
|ORD-1039|Kesehatan & Kecan...|          100000|
|ORD-1041|        Rumah Tangga|          300000|
|ORD-1042|             Fashion|         1250000|
|ORD-1048|   Makanan & Minuman|          200000|
|ORD-1053|          Elektronik|         3500000|
|ORD-1058|   Makanan & Minuman|          450000|
|ORD-1062|          Elektronik|           75000|
|ORD-1065|             Fashion|          450000|
|ORD-1068|          

In [2]:
from pyspark.sql.functions import sum, desc

df.groupBy("kategori") \
  .agg(sum("total_pendapatan").alias("total_pendapatan_kategori")) \
  .orderBy(desc("total_pendapatan_kategori")) \
  .show()

+--------------------+-------------------------+
|            kategori|total_pendapatan_kategori|
+--------------------+-------------------------+
|          Elektronik|                 97700000|
|   Makanan & Minuman|                 96375000|
|Kesehatan & Kecan...|                 92925000|
|        Rumah Tangga|                 92025000|
|             Fashion|                 88075000|
+--------------------+-------------------------+



In [3]:
df.groupBy("metode_pembayaran").count().show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD|  138|
|    Transfer Bank|  130|
|     Kartu Kredit|  164|
|         E-Wallet|  168|
+-----------------+-----+

